<a href="https://colab.research.google.com/github/mithunareddy/NLP/blob/main/Lab_Assignment_16_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
!pip install transformers datasets torch scikit-learn

In [19]:
from transformers import pipeline

In [20]:
classifier = pipeline("sentiment-analysis")
result = classifier("Hugging Face is very easy to use!")
print(result)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9988467693328857}]


In [21]:
from datasets import load_dataset
dataset = load_dataset("imdb")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [22]:
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
text = "Transformers are powerful models"
inputs = tokenizer(text,return_tensors="pt")
print(inputs)

{'input_ids': tensor([[  101, 19081,  2024,  3928,  4275,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}


In [24]:
outputs = model(**inputs)
cls_embedding = outputs.last_hidden_state[:, 0, :]
print(cls_embedding.shape)

torch.Size([1, 768])


In [25]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split

In [26]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [27]:
texts = [
    "I love this product",
    "This is amazing",
    "I am very happy",
    "I hate this",
    "This is bad",
    "Very disappointing"
]

labels = [1, 1, 1, 0, 0, 0]


In [28]:
def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    outputs = model(**inputs)

    cls_embedding = outputs.last_hidden_state[:, 0, :]

    return cls_embedding.detach().numpy()[0]

In [29]:
X = np.array([get_embedding(t) for t in texts])
y = np.array(labels)

print("Feature shape:", X.shape)

Feature shape: (6, 768)


In [30]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [31]:
clf = GaussianNB()
clf.fit(X_train, y_train)

GaussianNB()

In [32]:
y_pred = clf.predict(X_test)
print("Predictions:", y_pred)
print("Actual:", y_test)

Predictions: [1 0]
Actual: [0 1]


In [33]:
test_text = "This product is really good"

test_emb = get_embedding(test_text).reshape(1, -1)

prediction = clf.predict(test_emb)

print("Prediction:", "Positive" if prediction[0] == 1 else "Negative")

Prediction: Positive


In [34]:
import json
import IPython

IPython.display.clear_output()